# 05 — Train ST-GCN-inspired model on the full tracked UCF101 dataset

This is the corrected replacement for `05_train_stgcn.ipynb`.

It reads the full sequence dataset pointer written by Notebook 04. It does **not** modify Notebook 1, the extracted NPZ files, or the sequence arrays.

## Class support
The current dataset metadata reports 101 class names, but `BreastStroke` has no training windows. The model therefore trains only classes with actual training samples (expected: 100). Classes without validation windows do not contribute to validation macro-F1 checkpoint selection. Unsupported classes are reported explicitly; no samples are moved between splits.

## Training and recovery
- Batch size: 8
- CUDA required by default; training stops if the worker cannot access CUDA.
- Arrays are read through memory maps; DataLoader uses `num_workers=0` for Windows stability.
- A separate Python worker runs training.
- `latest_checkpoint.pt` is saved after every epoch; `stgcn_best.pt` tracks the best validation checkpoint; an additional checkpoint is saved every five epochs.
- Logs are written to disk rather than flooding the notebook output.
- Re-running the notebook against the same sequence build resumes from the latest checkpoint. Cell 2 checks for an already-running worker before launching another.

## Run order
Run Cells 1 and 2 once. Then rerun Cell 3 to monitor progress. Run Cells 4 and 5 after training has finished.


In [5]:
# CELL 1 — Verify full dataset and create the complete training worker
from pathlib import Path
import json, os, sys, subprocess, time
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
NOTEBOOK2_DIR = PROJECT_DIR / "Notebook_2_Improved_v2"
POINTER_PATH = NOTEBOOK2_DIR / "data" / "CURRENT_TRACKED_SEQUENCE_DIR.txt"

assert POINTER_PATH.is_file(), "Run the corrected full-dataset Notebook 04 first."
SEQUENCE_DIR = Path(POINTER_PATH.read_text(encoding="utf-8").strip())
assert SEQUENCE_DIR.is_dir(), f"Sequence directory not found: {SEQUENCE_DIR}"

META_PATH = SEQUENCE_DIR / "dataset_metadata.json"
assert META_PATH.is_file(), f"dataset_metadata.json not found: {META_PATH}"
META = json.loads(META_PATH.read_text(encoding="utf-8"))

assert META.get("is_full_dataset") is True, "The current pointer is not a full dataset."
assert int(META.get("expected_source_videos", 0)) == 13319, (
    f"Expected 13,319 source videos; metadata says {META.get('expected_source_videos')}."
)
assert int(META.get("tracked_source_videos", 0)) == 13319, (
    f"Not all source videos are represented: {META.get('tracked_source_videos')}."
)
assert float(META.get("source_video_coverage", 0)) == 1.0, (
    f"Coverage is not 100%: {META.get('source_video_coverage')}."
)

required_files = [
    f"{prefix}_{split}.npy"
    for split in ("train", "val", "test")
    for prefix in ("X", "y", "videos")
]
required_files += ["label_map.json", "class_support_by_split.csv"]
missing_files = [name for name in required_files if not (SEQUENCE_DIR / name).is_file()]
assert not missing_files, f"Missing sequence files: {missing_files}"

# Validate file headers using memory maps; do not load full arrays into RAM.
shapes = {}
for split in ("train", "val", "test"):
    x = np.load(SEQUENCE_DIR / f"X_{split}.npy", mmap_mode="r", allow_pickle=False)
    y = np.load(SEQUENCE_DIR / f"y_{split}.npy", mmap_mode="r", allow_pickle=False)
    videos = np.load(SEQUENCE_DIR / f"videos_{split}.npy", mmap_mode="r", allow_pickle=False)
    assert x.ndim == 3 and x.shape[1:] == (16, 51), f"Unexpected {split} X shape: {x.shape}"
    assert len(x) == len(y) == len(videos), f"{split} X/y/videos lengths differ."
    shapes[split] = tuple(x.shape)
    del x, y, videos

label_map = json.loads((SEQUENCE_DIR / "label_map.json").read_text(encoding="utf-8"))
class_support = pd.read_csv(SEQUENCE_DIR / "class_support_by_split.csv")
missing_train = class_support.loc[class_support["train_windows"].eq(0), "label"].astype(str).tolist()
missing_val = class_support.loc[class_support["val_windows"].eq(0), "label"].astype(str).tolist()
missing_test = class_support.loc[class_support["test_windows"].eq(0), "label"].astype(str).tolist()

print("FULL DATASET VERIFIED")
print("Sequence directory:", SEQUENCE_DIR)
print("Source videos:", META["tracked_source_videos"], "/", META["expected_source_videos"])
print("Sequence shapes:", shapes)
print("Original classes:", len(label_map))
print("Classes without train windows:", missing_train)
print("Classes without validation windows:", missing_val)
print("Classes without test windows:", missing_test)
print("Recording-group split reused:", META.get("recording_group_split_reused_without_changes"))

if min(shapes[s][0] for s in ("train", "val", "test")) == 0:
    raise RuntimeError("At least one split has zero windows. Training stopped.")

RUN_ID = SEQUENCE_DIR.name
RUN_DIR = NOTEBOOK2_DIR / "outputs" / "models" / f"stgcn_iou_v3_full_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

SRC_DIR = NOTEBOOK2_DIR / "src"
SRC_DIR.mkdir(parents=True, exist_ok=True)
WORKER_PATH = SRC_DIR / "train_stgcn_worker_v3.py"
CONFIG_PATH = RUN_DIR / "worker_config.json"

BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 9
SAVE_EVERY = 5
USE_CUDA = True
REQUIRE_CUDA = True
RESUME_IF_INTERRUPTED = True

if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available in this Jupyter kernel. Training was NOT started. "
        "Check torch.cuda.is_available() before continuing."
    )

print("Training GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
      if torch.cuda.is_available() else "N/A")

CONFIG = {
    "sequence_dir": str(SEQUENCE_DIR),
    "run_dir": str(RUN_DIR),
    "seed": 42,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "save_every": SAVE_EVERY,
    "use_cuda": USE_CUDA,
    "require_cuda": REQUIRE_CUDA,
    "resume": RESUME_IF_INTERRUPTED,
    "skip_if_complete": True,
    "learning_rate": 5e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.05,
}
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

WORKER_SOURCE = '# ST-GCN-inspired worker for the full tracked UCF101 sequence dataset.\n# This file is generated by 05_train_stgcn.ipynb and runs as a separate process.\n\nimport os\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("MKL_NUM_THREADS", "1")\nos.environ.setdefault("OPENBLAS_NUM_THREADS", "1")\nos.environ.setdefault("NUMEXPR_NUM_THREADS", "1")\n\nimport sys, json, time, random, traceback\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nfrom sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix\n\ndef log(message, log_path):\n    stamp = time.strftime("%Y-%m-%d %H:%M:%S")\n    line = f"[{stamp}] {message}"\n    print(line, flush=True)\n    with open(log_path, "a", encoding="utf-8") as f:\n        f.write(line + "\\n")\n\ndef atomic_torch_save(obj, path):\n    path = Path(path)\n    tmp = path.with_name(path.stem + ".tmp" + path.suffix)\n    torch.save(obj, tmp)\n    last_error = None\n    for attempt in range(10):\n        try:\n            os.replace(tmp, path)\n            return\n        except PermissionError as exc:\n            last_error = exc\n            time.sleep(min(attempt + 1, 3))\n    raise last_error\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n# COCO-17 skeleton graph, zero-based joint indices.\nEDGES = [\n    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),\n    (5, 11), (6, 12), (11, 12),\n    (11, 13), (13, 15), (12, 14), (14, 16),\n    (0, 1), (0, 2), (1, 3), (2, 4)\n]\nPARENTS = [0, 0, 0, 1, 2, 0, 0, 5, 6, 7, 8, 5, 6, 11, 12, 13, 14]\n\ndef make_adjacency():\n    n = 17\n    a = np.eye(n, dtype=np.float32)\n    for i, j in EDGES:\n        a[i, j] = 1.0\n        a[j, i] = 1.0\n    degree = a.sum(axis=1)\n    inv_sqrt = np.power(degree, -0.5)\n    return (inv_sqrt[:, None] * a * inv_sqrt[None, :]).astype(np.float32)\n\ndef build_motion_features(x):\n    # Input [B,T,51] = 17 joints * (x,y,confidence).\n    # Output [B,8,T,17]: x,y,confidence, velocity-x, velocity-y, speed, bone-x, bone-y.\n    b, t, _ = x.shape\n    pose = x.reshape(b, t, 17, 3)\n    xy = pose[..., :2]\n    conf = torch.clamp(torch.nan_to_num(pose[..., 2], nan=0.0), 0.0, 1.0)\n    vel = torch.zeros_like(xy)\n    if t > 1:\n        vel[:, 1:] = xy[:, 1:] - xy[:, :-1]\n    speed = torch.linalg.vector_norm(vel, dim=-1, keepdim=True)\n    parent_xy = xy[:, :, PARENTS, :]\n    bone = xy - parent_xy\n    feats = torch.cat([xy, conf.unsqueeze(-1), vel, speed, bone], dim=-1)\n    return feats.permute(0, 3, 1, 2).contiguous()\n\nclass WindowDataset(Dataset):\n    def __init__(self, x_path, y_path, video_path, class_id_to_local, allowed_old_ids):\n        self.x = np.load(x_path, mmap_mode="r", allow_pickle=False)\n        self.y = np.load(y_path, mmap_mode="r", allow_pickle=False)\n        self.videos = np.load(video_path, mmap_mode="r", allow_pickle=False)\n        allowed = np.isin(self.y, np.asarray(allowed_old_ids, dtype=self.y.dtype))\n        self.indices = np.flatnonzero(allowed).astype(np.int64)\n        self.class_id_to_local = class_id_to_local\n        self.y_old = np.asarray(self.y[self.indices], dtype=np.int64)\n        self.y_local = np.asarray([class_id_to_local[int(v)] for v in self.y_old], dtype=np.int64)\n        self.video_ids = np.asarray(self.videos[self.indices], dtype=np.int64)\n        if len(self.indices) == 0:\n            raise RuntimeError(f"No usable windows found in {x_path}")\n\n    def __len__(self):\n        return len(self.indices)\n\n    def __getitem__(self, index):\n        real_idx = int(self.indices[index])\n        sample = np.asarray(self.x[real_idx], dtype=np.float32)\n        return (\n            torch.from_numpy(sample),\n            torch.tensor(int(self.y_local[index]), dtype=torch.long),\n            torch.tensor(int(self.video_ids[index]), dtype=torch.long)\n        )\n\nclass STGCNBlock(nn.Module):\n    def __init__(self, in_channels, out_channels, adjacency, dropout=0.15):\n        super().__init__()\n        self.register_buffer("adjacency", torch.tensor(adjacency, dtype=torch.float32))\n        self.spatial = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)\n        self.spatial_bn = nn.BatchNorm2d(out_channels)\n        self.temporal = nn.Conv2d(out_channels, out_channels, kernel_size=(3, 1), padding=(1, 0), bias=False)\n        self.temporal_bn = nn.BatchNorm2d(out_channels)\n        self.dropout = nn.Dropout2d(dropout)\n        self.residual = (\n            nn.Identity() if in_channels == out_channels else nn.Sequential(\n                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),\n                nn.BatchNorm2d(out_channels)\n            )\n        )\n        self.activation = nn.ReLU(inplace=True)\n\n    def forward(self, x):\n        residual = self.residual(x)\n        x = self.activation(self.spatial_bn(self.spatial(x)))\n        x = torch.einsum("bctv,vw->bctw", x, self.adjacency)\n        x = self.temporal_bn(self.temporal(x))\n        x = self.dropout(x)\n        return self.activation(x + residual)\n\nclass STGCNInspired(nn.Module):\n    def __init__(self, num_classes, adjacency):\n        super().__init__()\n        self.block1 = STGCNBlock(8, 64, adjacency, dropout=0.10)\n        self.block2 = STGCNBlock(64, 96, adjacency, dropout=0.15)\n        self.block3 = STGCNBlock(96, 128, adjacency, dropout=0.20)\n        self.classifier = nn.Sequential(\n            nn.Linear(128, 192),\n            nn.ReLU(inplace=True),\n            nn.Dropout(0.40),\n            nn.Linear(192, num_classes)\n        )\n\n    def forward(self, x):\n        x = build_motion_features(x)\n        x = self.block1(x)\n        x = self.block2(x)\n        x = self.block3(x)\n        return self.classifier(x.mean(dim=(2, 3)))\n\ndef evaluate(model, loader, device, criterion=None):\n    model.eval()\n    losses, y_true, y_pred, probs, video_ids = [], [], [], [], []\n    with torch.no_grad():\n        for xb, yb, vb in loader:\n            xb = xb.to(device, non_blocking=(device.type == "cuda"))\n            yb = yb.to(device, non_blocking=(device.type == "cuda"))\n            logits = model(xb)\n            if criterion is not None:\n                losses.append(float(criterion(logits, yb).item()))\n            p = torch.softmax(logits, dim=1)\n            y_true.extend(yb.cpu().numpy().tolist())\n            y_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())\n            probs.extend(p.cpu().numpy().tolist())\n            video_ids.extend(vb.numpy().tolist())\n    y_true = np.asarray(y_true, dtype=np.int64)\n    y_pred = np.asarray(y_pred, dtype=np.int64)\n    probs = np.asarray(probs, dtype=np.float32)\n    video_ids = np.asarray(video_ids, dtype=np.int64)\n    observed = np.unique(y_true)\n    return {\n        "loss": float(np.mean(losses)) if losses else None,\n        "accuracy": float(accuracy_score(y_true, y_pred)),\n        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),\n        "macro_f1_supported_classes": float(f1_score(y_true, y_pred, labels=observed, average="macro", zero_division=0)),\n        "y_true": y_true, "y_pred": y_pred, "probabilities": probs,\n        "video_ids": video_ids, "observed_classes": observed,\n    }\n\ndef main(config_path):\n    config = json.loads(Path(config_path).read_text(encoding="utf-8"))\n    seq_dir = Path(config["sequence_dir"])\n    run_dir = Path(config["run_dir"])\n    run_dir.mkdir(parents=True, exist_ok=True)\n    log_path = run_dir / "training.log"\n    history_path = run_dir / "training_history.csv"\n    latest_path = run_dir / "latest_checkpoint.pt"\n    best_path = run_dir / "stgcn_best.pt"\n    metrics_path = run_dir / "test_metrics.json"\n\n    if config.get("skip_if_complete", True) and metrics_path.is_file() and best_path.is_file():\n        log("Final metrics and best checkpoint already exist. Skipping retraining.", log_path)\n        return 0\n\n    set_seed(int(config.get("seed", 42)))\n    device = torch.device("cuda:0" if config.get("use_cuda", True) and torch.cuda.is_available() else "cpu")\n    if config.get("require_cuda", True) and device.type != "cuda":\n        raise RuntimeError("CUDA required but unavailable in worker process.")\n    log(f"Worker device: {device}", log_path)\n    if device.type == "cuda":\n        log(f"GPU: {torch.cuda.get_device_name(0)}", log_path)\n        log(f"GPU memory GB: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f}", log_path)\n\n    label_map_raw = json.loads((seq_dir / "label_map.json").read_text(encoding="utf-8"))\n    class_names_by_old = {int(k): str(v) for k, v in label_map_raw.items()}\n\n    y_train_raw = np.load(seq_dir / "y_train.npy", mmap_mode="r", allow_pickle=False)\n    y_val_raw = np.load(seq_dir / "y_val.npy", mmap_mode="r", allow_pickle=False)\n    y_test_raw = np.load(seq_dir / "y_test.npy", mmap_mode="r", allow_pickle=False)\n    train_ids = np.unique(y_train_raw)\n    train_supported_old = sorted(int(v) for v in train_ids)\n    if not train_supported_old:\n        raise RuntimeError("No training classes found.")\n\n    old_to_local = {old: i for i, old in enumerate(train_supported_old)}\n    model_class_names = [class_names_by_old[old] for old in train_supported_old]\n    val_supported_old = sorted(int(v) for v in np.unique(y_val_raw) if int(v) in old_to_local)\n    test_supported_old = sorted(int(v) for v in np.unique(y_test_raw) if int(v) in old_to_local)\n    missing_train = [name for old, name in class_names_by_old.items() if old not in old_to_local]\n    missing_val = [class_names_by_old[old] for old in train_supported_old if old not in val_supported_old]\n    missing_test = [class_names_by_old[old] for old in train_supported_old if old not in test_supported_old]\n\n    log(f"Sequence counts: train={len(y_train_raw):,}, val={len(y_val_raw):,}, test={len(y_test_raw):,}", log_path)\n    log(f"Classes with training windows: {len(train_supported_old)}", log_path)\n    log(f"Classes without train windows (excluded): {missing_train}", log_path)\n    log(f"Train classes without validation windows: {missing_val}", log_path)\n    log(f"Train classes without test windows: {missing_test}", log_path)\n\n    train_ds = WindowDataset(seq_dir / "X_train.npy", seq_dir / "y_train.npy", seq_dir / "videos_train.npy", old_to_local, train_supported_old)\n    val_ds = WindowDataset(seq_dir / "X_val.npy", seq_dir / "y_val.npy", seq_dir / "videos_val.npy", old_to_local, train_supported_old)\n    test_ds = WindowDataset(seq_dir / "X_test.npy", seq_dir / "y_test.npy", seq_dir / "videos_test.npy", old_to_local, train_supported_old)\n    if len(val_ds) == 0 or len(test_ds) == 0:\n        raise RuntimeError("Validation or test has no windows from train-supported classes.")\n\n    batch_size = int(config.get("batch_size", 8))\n    pin_memory = device.type == "cuda"\n    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=pin_memory)\n    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)\n    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=pin_memory)\n\n    counts_local = np.bincount(train_ds.y_local, minlength=len(train_supported_old)).astype(np.float32)\n    class_weights = 1.0 / np.sqrt(np.maximum(counts_local, 1.0))\n    class_weights = class_weights / class_weights.mean()\n    criterion = nn.CrossEntropyLoss(\n        weight=torch.tensor(class_weights, dtype=torch.float32, device=device),\n        label_smoothing=float(config.get("label_smoothing", 0.05))\n    )\n    model = STGCNInspired(len(train_supported_old), make_adjacency()).to(device)\n    optimizer = torch.optim.AdamW(\n        model.parameters(), lr=float(config.get("learning_rate", 5e-4)),\n        weight_decay=float(config.get("weight_decay", 1e-4))\n    )\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6)\n\n    start_epoch, best_val_f1, best_val_loss, best_epoch, stale_epochs = 1, -1.0, float("inf"), 0, 0\n    history_rows = []\n    if config.get("resume", True) and latest_path.is_file():\n        checkpoint = torch.load(latest_path, map_location=device, weights_only=False)\n        if checkpoint.get("train_class_ids") != train_supported_old:\n            raise RuntimeError("Checkpoint class mapping differs from current dataset. Use a new run folder.")\n        model.load_state_dict(checkpoint["model_state"])\n        optimizer.load_state_dict(checkpoint["optimizer_state"])\n        if checkpoint.get("scheduler_state"):\n            scheduler.load_state_dict(checkpoint["scheduler_state"])\n        start_epoch = int(checkpoint["epoch"]) + 1\n        best_val_f1 = float(checkpoint.get("best_val_f1", -1.0))\n        best_val_loss = float(checkpoint.get("best_val_loss", float("inf")))\n        best_epoch = int(checkpoint.get("best_epoch", 0))\n        stale_epochs = int(checkpoint.get("stale_epochs", 0))\n        history_rows = checkpoint.get("history_rows", [])\n        log(f"Resuming from epoch {start_epoch - 1}.", log_path)\n\n    max_epochs = int(config.get("max_epochs", 50))\n    patience = int(config.get("patience", 9))\n    save_every = int(config.get("save_every", 5))\n\n    for epoch in range(start_epoch, max_epochs + 1):\n        model.train()\n        train_losses, train_true, train_pred = [], [], []\n        epoch_start = time.time()\n        for xb, yb, _ in train_loader:\n            xb = xb.to(device, non_blocking=pin_memory)\n            yb = yb.to(device, non_blocking=pin_memory)\n            optimizer.zero_grad(set_to_none=True)\n            logits = model(xb)\n            loss = criterion(logits, yb)\n            loss.backward()\n            nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)\n            optimizer.step()\n            train_losses.append(float(loss.item()))\n            train_true.extend(yb.detach().cpu().numpy().tolist())\n            train_pred.extend(logits.detach().argmax(dim=1).cpu().numpy().tolist())\n\n        train_true = np.asarray(train_true, dtype=np.int64)\n        train_pred = np.asarray(train_pred, dtype=np.int64)\n        train_seen = np.unique(train_true)\n        train_loss = float(np.mean(train_losses))\n        train_acc = float(accuracy_score(train_true, train_pred))\n        train_f1 = float(f1_score(train_true, train_pred, labels=train_seen, average="macro", zero_division=0))\n\n        val_result = evaluate(model, val_loader, device, criterion)\n        scheduler.step(val_result["loss"])\n        val_row = {\n            "epoch": epoch, "train_loss": train_loss, "train_accuracy": train_acc,\n            "train_macro_f1_supported": train_f1, "val_loss": val_result["loss"],\n            "val_accuracy": val_result["accuracy"], "val_balanced_accuracy": val_result["balanced_accuracy"],\n            "val_macro_f1_supported_classes": val_result["macro_f1_supported_classes"],\n            "learning_rate": float(optimizer.param_groups[0]["lr"]),\n            "epoch_seconds": round(time.time() - epoch_start, 2),\n        }\n        history_rows.append(val_row)\n        pd.DataFrame(history_rows).to_csv(history_path, index=False)\n\n        improved = (\n            val_result["macro_f1_supported_classes"] > best_val_f1 + 1e-6\n            or (abs(val_result["macro_f1_supported_classes"] - best_val_f1) <= 1e-6\n                and val_result["loss"] < best_val_loss)\n        )\n        if improved:\n            best_val_f1 = val_result["macro_f1_supported_classes"]\n            best_val_loss = float(val_result["loss"])\n            best_epoch = epoch\n            stale_epochs = 0\n        else:\n            stale_epochs += 1\n\n        state = {\n            "epoch": epoch, "model_state": model.state_dict(),\n            "optimizer_state": optimizer.state_dict(), "scheduler_state": scheduler.state_dict(),\n            "best_val_f1": best_val_f1, "best_val_loss": best_val_loss,\n            "best_epoch": best_epoch, "stale_epochs": stale_epochs, "history_rows": history_rows,\n            "train_class_ids": train_supported_old, "class_names": model_class_names,\n            "input_shape": [16, 51], "feature_channels": 8,\n            "architecture": "ST-GCN-inspired COCO-17 spatial-temporal graph network",\n        }\n        atomic_torch_save(state, latest_path)\n        if improved:\n            atomic_torch_save(state, best_path)\n        if epoch % save_every == 0:\n            atomic_torch_save(state, run_dir / f"checkpoint_epoch_{epoch:03d}.pt")\n\n        log(\n            f"Epoch {epoch:02d}/{max_epochs} | train loss={train_loss:.4f} acc={train_acc:.4f} F1={train_f1:.4f} | "\n            f"val loss={val_result[\'loss\']:.4f} acc={val_result[\'accuracy\']:.4f} "\n            f"bal_acc={val_result[\'balanced_accuracy\']:.4f} macro-F1={val_result[\'macro_f1_supported_classes\']:.4f} | "\n            f"best val F1={best_val_f1:.4f} epoch={best_epoch} | seconds={val_row[\'epoch_seconds\']}",\n            log_path\n        )\n        if device.type == "cuda":\n            torch.cuda.empty_cache()\n        if stale_epochs >= patience:\n            log(f"Early stopping at epoch {epoch}; no validation macro-F1 improvement for {patience} epochs.", log_path)\n            break\n\n    if not best_path.is_file():\n        raise RuntimeError("No best validation checkpoint was saved.")\n    best = torch.load(best_path, map_location=device, weights_only=False)\n    model.load_state_dict(best["model_state"])\n    test_result = evaluate(model, test_loader, device, criterion)\n    test_true, test_pred = test_result["y_true"], test_result["y_pred"]\n    observed_test = np.unique(test_true)\n\n    report = classification_report(\n        test_true, test_pred, labels=observed_test,\n        target_names=[model_class_names[int(i)] for i in observed_test],\n        output_dict=True, zero_division=0\n    )\n    pd.DataFrame(report).transpose().to_csv(run_dir / "test_classification_report.csv")\n\n    window_predictions = pd.DataFrame({\n        "true_class_id": [train_supported_old[int(i)] for i in test_true],\n        "true_label": [model_class_names[int(i)] for i in test_true],\n        "predicted_class_id": [train_supported_old[int(i)] for i in test_pred],\n        "predicted_label": [model_class_names[int(i)] for i in test_pred],\n        "correct": test_true == test_pred,\n        "video_id": test_result["video_ids"],\n    })\n    window_predictions.to_csv(run_dir / "test_window_predictions.csv", index=False)\n\n    cm = confusion_matrix(test_true, test_pred, labels=np.arange(len(train_supported_old)))\n    pd.DataFrame(cm, index=model_class_names, columns=model_class_names).to_csv(run_dir / "test_confusion_matrix.csv")\n\n    # Aggregate window probabilities per source video.\n    video_rows = []\n    probs, video_ids = test_result["probabilities"], test_result["video_ids"]\n    for vid in np.unique(video_ids):\n        idx = np.flatnonzero(video_ids == vid)\n        true_ids = np.unique(test_true[idx])\n        if len(true_ids) != 1:\n            raise RuntimeError(f"Video {vid} has multiple true labels.")\n        mean_prob = probs[idx].mean(axis=0)\n        pred_local, true_local = int(np.argmax(mean_prob)), int(true_ids[0])\n        video_rows.append({\n            "video_id": int(vid),\n            "true_class_id": train_supported_old[true_local],\n            "true_label": model_class_names[true_local],\n            "predicted_class_id": train_supported_old[pred_local],\n            "predicted_label": model_class_names[pred_local],\n            "correct": bool(pred_local == true_local),\n            "windows_aggregated": int(len(idx)),\n        })\n    video_df = pd.DataFrame(video_rows)\n    video_df.to_csv(run_dir / "test_video_predictions.csv", index=False)\n    video_true = video_df["true_class_id"].to_numpy()\n    video_pred = video_df["predicted_class_id"].to_numpy()\n    video_observed = np.unique(video_true)\n\n    metrics = {\n        "experiment": "ST-GCN-inspired model on full tracked UCF101 pose sequences",\n        "checkpoint": str(best_path),\n        "best_epoch": int(best_epoch),\n        "best_validation_macro_f1_supported_classes": float(best_val_f1),\n        "test_windows": int(len(test_true)),\n        "test_window_accuracy": float(accuracy_score(test_true, test_pred)),\n        "test_window_balanced_accuracy": float(balanced_accuracy_score(test_true, test_pred)),\n        "test_window_macro_f1_supported_classes": float(f1_score(\n            test_true, test_pred, labels=observed_test, average="macro", zero_division=0\n        )),\n        "test_source_videos": int(len(video_df)),\n        "test_video_accuracy": float(accuracy_score(video_true, video_pred)),\n        "test_video_balanced_accuracy": float(balanced_accuracy_score(video_true, video_pred)),\n        "test_video_macro_f1_supported_classes": float(f1_score(\n            video_true, video_pred, labels=video_observed, average="macro", zero_division=0\n        )),\n        "classes_in_model": int(len(train_supported_old)),\n        "classes_without_training_windows_excluded": missing_train,\n        "training_classes_without_validation_windows": missing_val,\n        "training_classes_without_test_windows": missing_test,\n        "test_classes_with_support": int(len(observed_test)),\n        "device": str(device),\n        "batch_size": batch_size,\n        "sequence_shape": [16, 51],\n        "architecture": "ST-GCN-inspired; 8 channels/joint from pose, confidence, velocity, speed and bone vectors",\n        "note": "Metrics are calculated over classes with support in the relevant split. Classes without training windows cannot be learned.",\n    }\n    (run_dir / "test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")\n    log("TRAINING COMPLETE", log_path)\n    log(json.dumps(metrics, indent=2), log_path)\n    log(f"Best checkpoint: {best_path}", log_path)\n    return 0\n\nif __name__ == "__main__":\n    if len(sys.argv) != 2:\n        print("Usage: train_stgcn_worker_v3.py <worker_config.json>", file=sys.stderr)\n        sys.exit(2)\n    try:\n        sys.exit(main(sys.argv[1]))\n    except Exception:\n        traceback.print_exc()\n        sys.exit(1)\n'
WORKER_PATH.write_text(WORKER_SOURCE, encoding="utf-8")
compile(WORKER_PATH.read_text(encoding="utf-8"), str(WORKER_PATH), "exec")

print("\nTraining worker created:", WORKER_PATH)
print("Training config:", CONFIG_PATH)
print("New model/report directory:", RUN_DIR)
print("Batch size:", BATCH_SIZE, "| max epochs:", MAX_EPOCHS, "| recovery every:", SAVE_EVERY, "epochs")
print("Generated worker syntax check: PASSED")


FULL DATASET VERIFIED
Sequence directory: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\data\sequences_tracked_group_split_iou_v3_full_v1\20261031_035536
Source videos: 13319 / 13319
Sequence shapes: {'train': (40404, 16, 51), 'val': (13424, 16, 51), 'test': (14452, 16, 51)}
Original classes: 101
Classes without train windows: ['BreastStroke']
Classes without validation windows: ['BreastStroke', 'FrontCrawl']
Classes without test windows: ['BreastStroke', 'CliffDiving']
Recording-group split reused: True
Training GPU: NVIDIA RTX A4000
GPU memory GB: 15.99

Training worker created: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\src\train_stgcn_worker_v3.py
Training config: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\outputs\models\stgcn_iou_v3_full_20261031_035536\worker_config.json
New model/report directory: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\outputs\models\stgcn_iou_v3_full_20261031_035536
Batch s

In [6]:
# CELL 2 — Launch the ST-GCN worker as an independent Windows process
# This cell starts exactly one worker and sends its verbose output to files.
import os, sys, subprocess
from pathlib import Path

PID_PATH = RUN_DIR / "training_worker.pid"
STDOUT_PATH = RUN_DIR / "worker_stdout.log"
STDERR_PATH = RUN_DIR / "worker_stderr.log"
TRAINING_LOG = RUN_DIR / "training.log"

if not WORKER_PATH.is_file() or not CONFIG_PATH.is_file():
    raise FileNotFoundError("Run Cell 1 successfully before launching training.")

def get_process_command_line(pid):
    # Use Windows CIM to distinguish this worker from an unrelated Python process.
    command = (
        f"$p = Get-CimInstance Win32_Process -Filter 'ProcessId = {int(pid)}' "
        f"-ErrorAction SilentlyContinue; if ($p) {{ $p.CommandLine }}"
    )
    try:
        result = subprocess.run(
            ["powershell", "-NoProfile", "-Command", command],
            capture_output=True, text=True, timeout=15
        )
        return (result.stdout or "").strip()
    except Exception:
        return ""

if PID_PATH.is_file():
    try:
        old_pid = int(PID_PATH.read_text(encoding="utf-8").strip())
    except Exception:
        old_pid = -1
    if old_pid > 0:
        old_cmd = get_process_command_line(old_pid)
        if old_cmd and "train_stgcn_worker_v3.py" in old_cmd:
            raise RuntimeError(
                f"ST-GCN worker PID {old_pid} is already running. "
                "Do not start a second worker. Run Cell 3 to monitor it."
            )

if (RUN_DIR / "test_metrics.json").is_file() and (RUN_DIR / "stgcn_best.pt").is_file():
    raise RuntimeError(
        "This run already has final metrics and a best checkpoint. "
        "Training will not be repeated. Inspect Cells 3 and 4."
    )

env = os.environ.copy()
torch_lib = Path(sys.prefix) / "Lib" / "site-packages" / "torch" / "lib"
if torch_lib.is_dir():
    env["PATH"] = str(torch_lib) + os.pathsep + env.get("PATH", "")
for key in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    env[key] = "1"
env["PYTHONUNBUFFERED"] = "1"

creationflags = 0
if os.name == "nt":
    creationflags = (
        getattr(subprocess, "DETACHED_PROCESS", 0)
        | getattr(subprocess, "CREATE_NEW_PROCESS_GROUP", 0)
    )

with STDOUT_PATH.open("w", encoding="utf-8") as stdout_file, STDERR_PATH.open("w", encoding="utf-8") as stderr_file:
    proc = subprocess.Popen(
        [sys.executable, str(WORKER_PATH), str(CONFIG_PATH)],
        cwd=str(PROJECT_DIR),
        env=env,
        stdin=subprocess.DEVNULL,
        stdout=stdout_file,
        stderr=stderr_file,
        creationflags=creationflags
    )

PID_PATH.write_text(str(proc.pid), encoding="utf-8")
print("ST-GCN training worker launched.")
print("Worker PID:", proc.pid)
print("Device requested: CUDA (worker verifies CUDA independently).")
print("Training log:", TRAINING_LOG)
print("Worker stdout:", STDOUT_PATH)
print("Worker stderr:", STDERR_PATH)
print("Run Cell 3 later to monitor. Do not launch Cell 2 twice.")


ST-GCN training worker launched.
Worker PID: 20280
Device requested: CUDA (worker verifies CUDA independently).
Training log: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\outputs\models\stgcn_iou_v3_full_20261031_035536\training.log
Worker stdout: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\outputs\models\stgcn_iou_v3_full_20261031_035536\worker_stdout.log
Worker stderr: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\outputs\models\stgcn_iou_v3_full_20261031_035536\worker_stderr.log
Run Cell 3 later to monitor. Do not launch Cell 2 twice.


In [16]:
# CELL 3 — Read-only training monitor (safe to rerun)
import subprocess, json
from pathlib import Path

PID_PATH = RUN_DIR / "training_worker.pid"
TRAINING_LOG = RUN_DIR / "training.log"
METRICS_PATH = RUN_DIR / "test_metrics.json"
STDERR_PATH = RUN_DIR / "worker_stderr.log"

pid = int(PID_PATH.read_text(encoding="utf-8").strip()) if PID_PATH.is_file() else -1
alive = False
if pid > 0:
    try:
        command = (
            f"$p = Get-CimInstance Win32_Process -Filter 'ProcessId = {pid}' "
            f"-ErrorAction SilentlyContinue; if ($p) {{ $p.CommandLine }}"
        )
        check = subprocess.run(
            ["powershell", "-NoProfile", "-Command", command],
            capture_output=True, text=True, timeout=15
        )
        alive = "train_stgcn_worker_v3.py" in (check.stdout or "")
    except Exception as exc:
        print("Could not query worker process:", repr(exc))

print("Worker PID:", pid)
print("Worker running:", alive)

if TRAINING_LOG.is_file():
    lines = TRAINING_LOG.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\nLatest training log:")
    for line in lines[-15:]:
        print(line)
else:
    print("Training log has not been created yet.")

if STDERR_PATH.is_file() and STDERR_PATH.stat().st_size:
    print("\nWorker stderr (last 4000 characters):")
    print(STDERR_PATH.read_text(encoding="utf-8", errors="replace")[-4000:])

if METRICS_PATH.is_file():
    print("\nFINAL TEST METRICS:")
    print(json.dumps(json.loads(METRICS_PATH.read_text(encoding="utf-8")), indent=2))
elif alive:
    print("\nTraining is running. Rerun this cell later to refresh status.")
else:
    print("\nNo final metrics file yet. Inspect worker_stdout.log and worker_stderr.log before restarting.")


Worker PID: 20280
Worker running: False

Latest training log:
  ],
  "training_classes_without_test_windows": [
    "CliffDiving"
  ],
  "test_classes_with_support": 99,
  "device": "cuda:0",
  "batch_size": 8,
  "sequence_shape": [
    16,
    51
  ],
  "architecture": "ST-GCN-inspired; 8 channels/joint from pose, confidence, velocity, speed and bone vectors",
  "note": "Metrics are calculated over classes with support in the relevant split. Classes without training windows cannot be learned."
}
[2026-10-31 05:03:04] Best checkpoint: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\outputs\models\stgcn_iou_v3_full_20261031_035536\stgcn_best.pt

Worker stderr (last 4000 characters):
e")
C:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\admin\AppData\Local\Programs\Python\Python313\Lib\site-pa

In [17]:
# CELL 4 — Final test metrics, class report and training history
import json
from pathlib import Path
import pandas as pd

METRICS_PATH = RUN_DIR / "test_metrics.json"
REPORT_PATH = RUN_DIR / "test_classification_report.csv"
SUPPORT_PATH = SEQUENCE_DIR / "class_support_by_split.csv"
HISTORY_PATH = RUN_DIR / "training_history.csv"

if METRICS_PATH.is_file():
    metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
    print("FINAL HELD-OUT TEST METRICS")
    print(json.dumps(metrics, indent=2))
else:
    print("Training has not produced final test metrics yet.")

if REPORT_PATH.is_file():
    print("\nPer-class test classification report:")
    display(pd.read_csv(REPORT_PATH, index_col=0))
else:
    print("Per-class report will appear after final evaluation.")

if HISTORY_PATH.is_file():
    print("\nTraining history (last 15 epochs):")
    display(pd.read_csv(HISTORY_PATH).tail(15))

if SUPPORT_PATH.is_file():
    support = pd.read_csv(SUPPORT_PATH)
    print("\nOriginal dataset class support by split:")
    display(support)


FINAL HELD-OUT TEST METRICS
{
  "experiment": "ST-GCN-inspired model on full tracked UCF101 pose sequences",
  "checkpoint": "C:\\Users\\admin\\ViTPose_Action_Recognition\\Notebook_2_Improved_v2\\outputs\\models\\stgcn_iou_v3_full_20261031_035536\\stgcn_best.pt",
  "best_epoch": 42,
  "best_validation_macro_f1_supported_classes": 0.41816637846791205,
  "test_windows": 14452,
  "test_window_accuracy": 0.5375726543039026,
  "test_window_balanced_accuracy": 0.43326131323129324,
  "test_window_macro_f1_supported_classes": 0.4130954994925104,
  "test_source_videos": 1943,
  "test_video_accuracy": 0.5301080802882141,
  "test_video_balanced_accuracy": 0.46486086117131303,
  "test_video_macro_f1_supported_classes": 0.44706360024066955,
  "classes_in_model": 100,
  "classes_without_training_windows_excluded": [
    "BreastStroke"
  ],
  "training_classes_without_validation_windows": [
    "FrontCrawl"
  ],
  "training_classes_without_test_windows": [
    "CliffDiving"
  ],
  "test_classes_with_

,precision,recall,f1-score,support
ApplyEyeMakeup,0.338843,0.623574,0.439090,263.000000
ApplyLipstick,0.210300,0.264865,0.234450,185.000000
Archery,0.225806,0.260870,0.242075,161.000000
BabyCrawling,0.678082,0.846154,0.752852,117.000000
BalanceBeam,0.166667,0.035714,0.058824,28.000000
...,...,...,...,...
WritingOnBoard,0.807292,0.685841,0.741627,226.000000
YoYo,0.630208,0.584541,0.606516,207.000000
accuracy,0.537573,0.537573,0.537573,0.537573
macro avg,0.425819,0.433261,0.413095,14452.000000



Training history (last 15 epochs):


,epoch,train_loss,train_accuracy,train_macro_f1_supported,val_loss,val_accuracy,val_balanced_accuracy,val_macro_f1_supported_classes,learning_rate,epoch_seconds
35,36,1.515082,0.834200,0.700666,2.643731,0.533820,0.434768,0.411416,0.000016,45.80
36,37,1.515049,0.834992,0.703240,2.629964,0.538588,0.438671,0.416634,0.000008,46.43
37,38,1.512155,0.834398,0.704274,2.741066,0.529350,0.432088,0.407811,0.000008,44.74
38,39,1.508758,0.835338,0.711841,2.641943,0.536651,0.436970,0.415366,0.000008,44.77
39,40,1.509773,0.835907,0.701064,2.645433,0.533224,0.436593,0.413315,0.000008,43.86
40,41,1.510884,0.834200,0.707229,2.638612,0.535608,0.439406,0.418060,0.000004,45.77
41,42,1.504884,0.834917,0.709211,2.634898,0.539482,0.439147,0.418166,0.000004,46.20
42,43,1.504632,0.834447,0.706766,2.703649,0.533373,0.431090,0.408586,0.000004,48.01
43,44,1.511182,0.835239,0.704562,2.696398,0.528159,0.432705,0.408864,0.000004,47.86
44,45,1.507960,0.836006,0.708862,2.655749,0.535980,0.438443,0.416030,0.000002,45.98



Original dataset class support by split:


,class_id,label,train_windows,train_videos,val_windows,val_videos,test_windows,test_videos
0,0,ApplyEyeMakeup,677,88,206,27,263,30
1,1,ApplyLipstick,577,69,167,21,185,24
2,2,Archery,678,81,164,28,161,22
3,3,BabyCrawling,434,68,123,22,117,20
4,4,BalanceBeam,68,23,12,4,28,8
...,...,...,...,...,...,...,...,...
96,96,VolleyballSpiking,137,46,35,15,29,13
97,97,WalkingWithDog,512,69,102,20,202,24
98,98,WallPushups,311,76,114,24,134,28
99,99,WritingOnBoard,653,88,235,31,226,31


In [18]:
# CELL 5 — Model files and interpretation
print("Run directory:", RUN_DIR)
for name in [
    "stgcn_best.pt",
    "latest_checkpoint.pt",
    "training_history.csv",
    "test_metrics.json",
    "test_classification_report.csv",
    "test_confusion_matrix.csv",
    "test_window_predictions.csv",
    "test_video_predictions.csv",
    "training.log",
    "worker_stdout.log",
    "worker_stderr.log",
]:
    path = RUN_DIR / name
    print(f"{name}: {'PRESENT' if path.is_file() else 'not created yet'}")

print("\nInterpretation:")
print("- This is an ST-GCN-inspired COCO-17 spatial-temporal graph model, not an exact CTR-GCN reproduction.")
print("- It trains only on classes with training windows. Your current split has no BreastStroke training windows.")
print("- Validation checkpoint selection uses classes represented in validation.")
print("- Test metrics use the supported test classes and report their number explicitly.")
print("- Window-level and source-video-level scores are reported separately.")
print("- The original sequence arrays and Notebook 1 checkpoints are read-only.")


Run directory: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\outputs\models\stgcn_iou_v3_full_20261031_035536
stgcn_best.pt: PRESENT
latest_checkpoint.pt: PRESENT
training_history.csv: PRESENT
test_metrics.json: PRESENT
test_classification_report.csv: PRESENT
test_confusion_matrix.csv: PRESENT
test_window_predictions.csv: PRESENT
test_video_predictions.csv: PRESENT
training.log: PRESENT
worker_stdout.log: PRESENT
worker_stderr.log: PRESENT

Interpretation:
- This is an ST-GCN-inspired COCO-17 spatial-temporal graph model, not an exact CTR-GCN reproduction.
- It trains only on classes with training windows. Your current split has no BreastStroke training windows.
- Validation checkpoint selection uses classes represented in validation.
- Test metrics use the supported test classes and report their number explicitly.
- Window-level and source-video-level scores are reported separately.
- The original sequence arrays and Notebook 1 checkpoints are read-only.


## Important
The screenshot error `NameError: name 'WORKER_SCRIPT_PLACEHOLDER' is not defined` came from an incomplete worker template in the previous file. This replacement embeds the complete worker source directly in Cell 1 and compile-checks it before launch.

Do not rerun Notebook 03 or Notebook 04. Your full sequence arrays are already built. If Cell 1 reports a missing file or CUDA is unavailable, stop and share that exact message rather than changing packages or paths.
